### Import

In [41]:
import sys
import time
from time import perf_counter
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import mlflow

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

from src.evaluation.metrics_report import evaluate_model

In [42]:
df = pd.read_csv("../data/processed/rossmannV2.csv")

In [43]:
# mlflow.set_tracking_uri("http://127.0.0.1:5000")
# mlflow.set_experiment("rossmann-forecasting")

---

### Splitting the data and Metric choosing

In [44]:
df = df.sort_values("Date").reset_index(drop=True)

cutoff = "2015-01-01"

train = df[df["Date"] < cutoff].copy()
test = df[df["Date"] >= cutoff].copy()

In [45]:
X_train = train.drop(columns=["Sales", "Date"])
y_train = train["Sales"]

X_test = test.drop(columns=["Sales", "Date"])
y_test = test["Sales"]

In [46]:
tscv = TimeSeriesSplit(
    n_splits=5,
    gap=7
)

Here i added gap for so i have 7 rows gap between splits but it'll split wrong several times since dataset has several stores.

In [47]:
for train_idx, valid_idx in tscv.split(X_train):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_valid = X_train.iloc[valid_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_valid = y_train.iloc[valid_idx]

Prevents future data leaking.

---

Also i decided to use RMSLE for final model since it penalizes relative errors and is appropriate for sales with large variation.

But for model selection I'll use MAE, RMSE and RMSLE.

---

### Baseline and Model comparison

In [48]:
ridge_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("model", Ridge(alpha=1.0))
])

In [49]:
start = time.perf_counter()

ridge_pipeline.fit(X_fold_train, y_fold_train)

ridge_predict = ridge_pipeline.predict(X_fold_valid)

# Sales can't be negative, so i clip predictions at 0 before computing RMSLE
ridge_predict = np.clip(ridge_predict, 0, None)

print(evaluate_model(y_fold_valid, ridge_predict))
print(f"Time: {perf_counter() - start:.4f}s.")

{'MAE': 826.2660765644697, 'RMSE': 1171.0625652210192, 'RMSLE': 1.9011912212526134}
Time: 2.2518s.


---

In [50]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=44
)

start = time.perf_counter()

xgb_model.fit(X_fold_train, y_fold_train)

xgb_predict = xgb_model.predict(X_fold_valid)

print(evaluate_model(y_fold_valid, xgb_predict))
print(f"Time: {perf_counter() - start:.4f}s.")

{'MAE': 446.8831481933594, 'RMSE': 674.0966186523438, 'RMSLE': 1.325823426246643}
Time: 17.8234s.


---

In [52]:
lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=46,
    verbose=-1
)

start = time.perf_counter()

lgbm_model.fit(X_fold_train, y_fold_train)

lgbm_predict = lgbm_model.predict(X_fold_valid)

print(evaluate_model(y_fold_valid, lgbm_predict))
print(f"Time: {perf_counter() - start:.4f}s.")

{'MAE': 458.4409893311257, 'RMSE': 695.1642749861792, 'RMSLE': 1.1521239970014348}
Time: 12.2377s.


Here's the conclusions based on the results:
- XGBoost and LGBM did almost `50% better` than baseline, but Ridge was 5-8 times faster than complex models

- LGBM finished work `45% faster` than XGB

- On the other side XGB did `6% better than LGBM on average`, which is not significant difference

Because of LGBM speed and high scores I'll use it as a final model.

---

### Hyperparams + Best model